In [4]:
import plotly.io as pio
pio.renderers.default = "notebook_connected+png" 

from IPython.display import Image, display
import sys
import os
# Add project root to path
sys.path.append(os.path.abspath(".."))
# libs 
import numpy as np
import plotly.graph_objects as go
import requests
import pandas as pd
import ast
import time

# import data market from polymarket 
from src.analytics.download_data import safe_parse_list, fetch_many_markets, filter_fed_rates_markets, extract_yes_token_id, get_price_history




In [5]:
# 1. Download closed markets

df_markets_raw = fetch_many_markets(
    max_pages=10,
    limit=500,
    closed=True
)

print("Total markets downloaded:", len(df_markets_raw))

# 2. Filter Fed / rates markets


df_fed = filter_fed_rates_markets(df_markets_raw)

df_fed["yes_token_id"] = df_fed.apply(extract_yes_token_id, axis=1)

cols_to_keep = [
    "id",
    "question",
    "slug",
    "category",
    "endDate",
    "closed",
    "volume",
    "liquidity",
    "outcomes",
    "outcomePrices",
    "clobTokenIds",
    "yes_token_id"
]

cols_to_keep = [c for c in cols_to_keep if c in df_fed.columns]

df_fed = df_fed[cols_to_keep].dropna(subset=["yes_token_id"])

print("Fed/rates markets found:", len(df_fed))

display(df_fed.head())



# 3. Download price history
all_histories = []

for _, row in df_fed.iterrows():
    token_id = row["yes_token_id"]

    try:
        hist = get_price_history(
            clob_token_id=token_id,
            fidelity_minutes=60
        )

        if len(hist) > 0:
            hist["market_id"] = row["id"]
            hist["question"] = row.get("question", None)
            hist["slug"] = row.get("slug", None)
            all_histories.append(hist)

        time.sleep(0.25)

    except Exception as e:
        print(f"Error for market {row.get('id')}: {e}")


df_prices = pd.concat(all_histories, ignore_index=True) if all_histories else pd.DataFrame()

print("Price history rows:", len(df_prices))

display(df_prices.head())

Total markets downloaded: 1000
Fed/rates markets found: 70


,id,question,slug,category,endDate,closed,volume,liquidity,outcomes,outcomePrices,clobTokenIds,yes_token_id
14,57,Will Dharma’s Phase 1 Retroactive UNI Distribu...,will-dharma-s-phase-1-retroactive-uni-distribu...,Crypto,2020-10-31T00:00:00Z,True,35661.941003,0,"[""Yes"", ""No""]","[""0"", ""0""]","[""72218237979282777228049705424647682506448885...",7221823797928277722804970542464768250644888572...
15,59,Will there be a federal charge filed against H...,will-there-be-a-federal-charge-filed-against-h...,US-current-affairs,2021-01-02T00:00:00Z,True,127476.154269,0.13149,"[""Yes"", ""No""]","[""0.0000002365760136311761420505896616707271"",...","[""13335616915502503908960482263427811070742976...",1333561691550250390896048226342781107074297618...
25,73,Will Donald Trump tweet announcing that he won...,will-donald-trump-tweet-announcing-that-he-won...,US-current-affairs,2020-11-05T00:00:00Z,True,127490.669491,3.295904,"[""Yes"", ""No""]","[""0.000001969126198486787442291877571524744"", ...","[""76182894856278632248939743050168442512633640...",7618289485627863224893974305016844251263364038...
27,75,Will the Ethereum 2.0 Genesis Event happen suc...,will-the-ethereum-20-genesis-event-happen-succ...,Crypto,2020-12-02T00:00:00Z,True,560270.423771,7.877976,"[""Yes"", ""No""]","[""0.9999991353509498666949062749530927"", ""0.00...","[""77627371401939105002385145056688905915571707...",7762737140193910500238514505668890591557170756...
29,77,Will Donald Trump formally concede the 2020 US...,will-donald-trump-formally-concede-the-2020-us...,US-current-affairs,2020-12-01T00:00:00Z,True,454342.990794,8688.076983,"[""Yes"", ""No""]","[""0.0000001590260745451327528785858607902578"",...","[""73163434197768416862075961184775012109464929...",7316343419776841686207596118477501210946492901...


Error for market 57: 400 Client Error: Bad Request for url: https://clob.polymarket.com/prices-history?market=72218237979282777228049705424647682506448885729323094146056564018197398411018&fidelity=60
Error for market 59: 400 Client Error: Bad Request for url: https://clob.polymarket.com/prices-history?market=1333561691550250390896048226342781107074297618285636393912534214135261982511&fidelity=60
Error for market 73: 400 Client Error: Bad Request for url: https://clob.polymarket.com/prices-history?market=76182894856278632248939743050168442512633640389366255006450765930375044831481&fidelity=60
Error for market 75: 400 Client Error: Bad Request for url: https://clob.polymarket.com/prices-history?market=77627371401939105002385145056688905915571707561396568073129144307944860669891&fidelity=60
Error for market 77: 400 Client Error: Bad Request for url: https://clob.polymarket.com/prices-history?market=73163434197768416862075961184775012109464929015046649243834802132939776088568&fidelity=60
E

""
